# F2LLM-v2-330M  :  Fine-tuning MTEB French

**Objectif :** Fine-tuner `codefuse-ai/F2LLM-v2-330M` sur MIRACL-fr avec MNR Loss pour améliorer le score MTEB French.  
**Hardware cible :** RTX 3070 (8 GB VRAM)  
**Approche :** Transformers natif (pas sentence-transformers), fidèle au model card officiel.

## 0. Imports & vérifications

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TQDM_DISABLE"] = "0"

# Import tqdm std en premier, avant que notebook puisse s'enregistrer
from tqdm import tqdm
from tqdm.std import tqdm as tqdm_std

In [2]:
import os
import json

import torch
import torch.nn as nn
import torch.nn.functional as F

import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import OneCycleLR
from transformers import AutoTokenizer, AutoModel
from peft import get_peft_model, LoraConfig, TaskType
print(f"PyTorch   : {torch.__version__}")
print(f"CUDA dispo: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU       : {torch.cuda.get_device_name(0)}")
    print(f"VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device    : {DEVICE}")

PyTorch   : 2.5.1+cu121
CUDA dispo: True
GPU       : NVIDIA GeForce RTX 3070
VRAM      : 8.6 GB
Device    : cuda


## 1. Modèle  :  F2LLMEmbedder

Implémentation fidèle au [model card officiel](https://huggingface.co/codefuse-ai/F2LLM-v2-330M) :  
- Embedding = dernier token EOS du `last_hidden_state`  
- Normalisation L2 systématique  
- `encode()` avec `no_grad` pour l'inférence  
- `forward()` avec grad pour le training

In [3]:
class F2LLMEmbedder(nn.Module):

    def __init__(self, model_name: str = "codefuse-ai/F2LLM-v2-330M"):
        super().__init__()

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True
        )
        self.model = AutoModel.from_pretrained(
            model_name,
            trust_remote_code=True,
            torch_dtype=torch.bfloat16   # bfloat16 : 2x plus rapide sur RTX 3070, sans perte
        )

    # ------------------------------------------------------------------
    # Inférence (no_grad)  :  pour évaluation et vérification
    # ------------------------------------------------------------------
    def encode(self, texts, device=None):
        """Encode une liste de textes → embeddings normalisés (B, hidden_dim)."""

        if isinstance(texts, str):
            texts = [texts]

        if device is None:
            device = next(self.model.parameters()).device

        batch = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = self.model(**batch)

        return self._pool(outputs.last_hidden_state, batch["attention_mask"])

    # ------------------------------------------------------------------
    # Training (avec grad)  :  appelé dans la boucle d'entraînement
    # ------------------------------------------------------------------
    def forward(self, texts):
        """Forward pass avec gradient  :  retourne embeddings normalisés (B, hidden_dim)."""

        device = next(self.model.parameters()).device

        batch = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        outputs = self.model(**batch)

        return self._pool(outputs.last_hidden_state, batch["attention_mask"])

    # ------------------------------------------------------------------
    # Pooling interne  :  dernier token EOS + L2 normalize
    # Fidèle au model card : eos_positions = attention_mask.sum(dim=1) - 1
    # ------------------------------------------------------------------
    def _pool(self, last_hidden_state, attention_mask):
        B      = last_hidden_state.size(0)
        device = last_hidden_state.device

        eos_pos = attention_mask.sum(dim=1) - 1              # (B,)
        emb     = last_hidden_state[torch.arange(B, device=device), eos_pos]  # (B, D)
        emb     = F.normalize(emb, p=2, dim=1)

        return emb

In [4]:
# Chargement et vérification rapide
model = F2LLMEmbedder("codefuse-ai/F2LLM-v2-330M").to(DEVICE)

test_emb = model.encode(["bonjour", "comment ça va ?"])
print(f"Shape     : {test_emb.shape}")                        # (2, 896)
print(f"Normes    : {test_emb.norm(dim=1)}")                  # [1.0, 1.0]
print(f"dtype     : {test_emb.dtype}")                        # bfloat16
print(f"requires_grad (encode) : {test_emb.requires_grad}")   # False

test_fwd = model.forward(["bonjour"])
print(f"requires_grad (forward): {test_fwd.requires_grad}")   # True

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/178 [00:00<?, ?it/s]

f:\codeAI\llm\.venv\lib\site-packages\torch\nn\modules\module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


Shape     : torch.Size([2, 896])
Normes    : tensor([0.9961, 1.0000], device='cuda:0', dtype=torch.bfloat16)
dtype     : torch.bfloat16
requires_grad (encode) : False
requires_grad (forward): True


### LORA

In [5]:
# CELL — Injection LoRA sur model.model
lora_config = LoraConfig(
    r=16,                  # rang : plus élevé = plus expressif mais plus de VRAM
    lora_alpha=32,         # scaling = 2×r, standard
    target_modules=[       # couches attention du Qwen3 (backbone F2LLM)
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
)

model.model = get_peft_model(model.model, lora_config)
model.model.print_trainable_parameters()
# Attendu : ~1-2% des paramètres → ~3-6M au lieu de 330M
# Effet direct : VRAM optimizer states divisée par ~50

# Vérification que le pooling fonctionne toujours
test = model.encode(["bonjour"])
print(f"Shape : {test.shape}")   # (1, 896)
print(f"Norme : {test.norm():.4f}")  # ~1.0

trainable params: 5,144,576 || all params: 339,493,760 || trainable%: 1.5154
Shape : torch.Size([1, 896])
Norme : 1.0000


## 2. Dataset  :  MIRACL French

On utilise le dataset complet (pas le subset 100) pour un fine-tuning sérieux.

In [6]:
ds = load_dataset(
    "sentence-transformers/miracl",
    "fr-triplet-all",   # dataset complet français
    split="train"
)

print(f"Exemples      : {len(ds)}")
print(f"Colonnes      : {ds.column_names}")
print()
print("--- Exemple ---")
print("anchor  :", ds[0]["anchor"][:120])
print("positive:", ds[0]["positive"][:120])

Exemples      : 232100
Colonnes      : ['anchor', 'positive', 'negative']

--- Exemple ---
anchor  : Qui était Princesse Diana?
positive: Diana Spencer
La « dianamania » bat son plein les premières années qui suivent la mort de la princesse de Galles, mais s


## 3. Dataset PyTorch : paires (anchor, positive)

MNR Loss n'a besoin que de paires : pas de négatifs explicites.  
Les négatifs viennent des autres exemples du même batch (in-batch negatives).

In [7]:
class PairDataset(Dataset):

    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        return {
            "anchor"  : row["anchor"],
            "positive": row["positive"]
        }


train_dataset = PairDataset(ds)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,     # batch élevé = plus de négatifs in-batch = meilleur signal MNR
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

print(f"Dataset   : {len(train_dataset)} exemples")
print(f"Batches   : {len(train_loader)}")

# Vérification d'un batch
batch = next(iter(train_loader))
print(f"Clés batch: {list(batch.keys())}")
print(f"Anchor[0] : {batch['anchor'][0][:80]}")

Dataset   : 232100 exemples
Batches   : 7254
Clés batch: ['anchor', 'positive']
Anchor[0] : Comment devenir enseignant dans un collège en France?


## 4. Loss  :  Multiple Negatives Ranking (MNR)

Standard SOTA pour les embedding models. Principe :  
- Pour un batch de B paires (aᵢ, pᵢ)  
- On calcule la matrice de similarité B×B entre tous les anchors et tous les positifs  
- La diagonale doit être maximisée (aᵢ ↔ pᵢ)  
- Tous les autres positifs du batch deviennent des négatifs implicites  
- `temperature=0.05` : calibré pour embeddings normalisés (dot product ∈ [-1, 1])

In [8]:
def mnr_loss(anchor_emb: torch.Tensor, positive_emb: torch.Tensor, temperature: float = 0.05) -> torch.Tensor:
    """
    Multiple Negatives Ranking Loss (InfoNCE).

    Args:
        anchor_emb   : (B, D)  :  embeddings L2-normalisés des anchors
        positive_emb : (B, D)  :  embeddings L2-normalisés des positifs
        temperature  : scaling de la matrice de similarité

    Returns:
        loss scalaire
    """
    # Dot product = cosine similarity (embeddings normalisés)
    # sim_matrix[i][j] = sim(anchor_i, positive_j)
    sim_matrix = anchor_emb @ positive_emb.T / temperature  # (B, B)

    # Labels : anchor_i doit matcher positive_i → diagonale
    labels = torch.arange(sim_matrix.size(0), device=sim_matrix.device)

    return F.cross_entropy(sim_matrix, labels)


# Test sur un mini-batch
with torch.no_grad():
    a = model.encode(["Quelle est la capitale de la France ?"])
    p = model.encode(["Paris est la capitale de la France."])
    n = model.encode(["Le bœuf bourguignon est un plat traditionnel."])

# Cast float32 pour le test de loss (bfloat16 peut causer des issues avec cross_entropy)
a, p, n = a.float(), p.float(), n.float()

loss_good = mnr_loss(a, p)
loss_bad  = mnr_loss(a, n)
print(f"Loss (anchor ↔ positif) : {loss_good.item():.4f}")
print(f"Loss (anchor ↔ négatif) : {loss_bad.item():.4f}")  # doit être > loss_good

Loss (anchor ↔ positif) : 0.0000
Loss (anchor ↔ négatif) : 0.0000


In [9]:
# Test MNR sur un mini-batch réaliste (pas des singletons)
test_texts_a = [
    "Quelle est la capitale de la France ?",
    "Comment fonctionne un moteur électrique ?",
    "Quels sont les symptômes du diabète ?",
    "Qui a écrit Les Misérables ?",
]
test_texts_p = [
    "Paris est la capitale et la plus grande ville de France.",
    "Un moteur électrique convertit l'énergie électrique en énergie mécanique via un champ magnétique.",
    "Les symptômes incluent une soif excessive, une fatigue et des urinations fréquentes.",
    "Les Misérables est un roman de Victor Hugo publié en 1862.",
]

with torch.no_grad():
    ea = model.encode(test_texts_a).float()  # (4, 896)
    ep = model.encode(test_texts_p).float()  # (4, 896)

loss_val = mnr_loss(ea, ep, temperature=0.05)

# Matrice de similarité pour visualiser
sim_matrix = (ea @ ep.T).cpu()
print("Matrice de similarité (4×4) :")
print(sim_matrix.round(decimals=3))
print()
print(f"Loss MNR (B=4) : {loss_val.item():.4f}")
# Attendu : diagonale >> hors-diagonale, loss < log(4) ≈ 1.386
print(f"Loss random ref: {torch.log(torch.tensor(4.0)).item():.4f}  ← loss si modèle aléatoire")

Matrice de similarité (4×4) :
tensor([[ 0.5990,  0.0730,  0.0090,  0.1170],
        [ 0.0920,  0.5750, -0.0120,  0.0240],
        [ 0.1170,  0.0870,  0.3120,  0.1480],
        [ 0.1290,  0.0490,  0.0020,  0.5200]])

Loss MNR (B=4) : 0.0170
Loss random ref: 1.3863  ← loss si modèle aléatoire


## 5. Setup entraînement

- **AdamW** avec weight decay  
- **OneCycleLR** : 10% warmup linéaire puis cosine decay  :  standard pour fine-tuning LLM  
- **Gradient clipping** : stabilise les premiers steps

In [10]:
import subprocess

def get_gpu_temp():
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=temperature.gpu", "--format=csv,noheader"],
            encoding="utf-8"
        )
        return result.strip()
    except:
        return "N/A"

In [11]:
NUM_EPOCHS   = 3
LR           = 2e-5
TEMPERATURE  = 0.05
WARMUP_FRAC  = 0.1     # 10% des steps en warmup
MAX_GRAD_NORM = 1.0

TOTAL_STEPS  = len(train_loader) * NUM_EPOCHS
WARMUP_STEPS = int(TOTAL_STEPS * WARMUP_FRAC)
model.model.gradient_checkpointing_enable()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=0.01
)

scheduler = OneCycleLR(
    optimizer,
    max_lr=LR,
    total_steps=TOTAL_STEPS,
    pct_start=WARMUP_FRAC,
    anneal_strategy="cos"
)

print(f"Total steps  : {TOTAL_STEPS}")
print(f"Warmup steps : {WARMUP_STEPS}")
print(f"LR max       : {LR}")

Total steps  : 21762
Warmup steps : 2176
LR max       : 2e-05


## 6. Boucle d'entraînement

In [12]:
import sys
import rich
from rich.live import Live
from rich.table import Table
from rich.console import Console

console = Console()
os.makedirs("checkpoints", exist_ok=True)

ACCUM_STEPS = 4
PRINT_EVERY = 50  # toutes les 50 updates

history = []
model.train()

for epoch in range(NUM_EPOCHS):

    total_loss = 0.0
    n_batches  = 0
    n_updates  = 0

    optimizer.zero_grad()

    # Table rich mise à jour en live
    with Live(console=console, refresh_per_second=1) as live:

        for step, batch in enumerate(train_loader):

            anchor_emb   = model.forward(list(batch["anchor"]))
            positive_emb = model.forward(list(batch["positive"]))

            loss = mnr_loss(
                anchor_emb.float(),
                positive_emb.float(),
                temperature=TEMPERATURE
            )

            loss = loss / ACCUM_STEPS
            loss.backward()

            total_loss += loss.item() * ACCUM_STEPS
            n_batches  += 1

            if (step + 1) % ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                n_updates += 1

                if n_updates % PRINT_EVERY == 0:
                    avg   = total_loss / n_batches
                    vram  = torch.cuda.memory_allocated() / 1e9
                    pct   = (step + 1) / len(train_loader) * 100
                    lr    = scheduler.get_last_lr()[0]

                    # Tableau rich mis à jour en temps réel
                    table = Table(title=f"Epoch {epoch+1}/{NUM_EPOCHS}")
                    table.add_column("Metric",  style="cyan",  width=20)
                    table.add_column("Value",   style="green", width=20)

                    table.add_row("Progress",   f"{pct:.1f}%  ({step+1}/{len(train_loader)})")
                    table.add_row("Update",     f"{n_updates}")
                    table.add_row("Loss",       f"{avg:.4f}")
                    table.add_row("LR",         f"{lr:.2e}")
                    table.add_row("VRAM",       f"{vram:.2f} GB")
                    table.add_row("Temp GPU",   f"{get_gpu_temp()}°C")

                    live.update(table)

    avg_loss = total_loss / n_batches
    history.append({"epoch": epoch + 1, "avg_loss": avg_loss})

    console.print(f"\n✅ Epoch {epoch+1} terminée | loss : {avg_loss:.4f}")

    ckpt_path = f"checkpoints/f2llm_fr_epoch{epoch+1}.pt"
    torch.save(model.state_dict(), ckpt_path)
    console.print(f"💾 Checkpoint : {ckpt_path}\n")

console.print("🎉 Entraînement terminé.")

Output()

KeyboardInterrupt: 

## 7. Courbe de loss

In [ ]:
import matplotlib.pyplot as plt

df_history = pd.DataFrame(history)

plt.figure(figsize=(7, 4))
plt.plot(df_history["epoch"], df_history["avg_loss"], marker="o", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("MNR Loss (moyenne)")
plt.title("F2LLM-v2-330M  :  Fine-tuning MIRACL-fr")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_loss.png", dpi=150)
plt.show()

print(df_history)

## 8. Vérification qualitative post-training

In [ ]:
model.eval()

pairs = [
    (
        "Quelle est la capitale de la France ?",
        "Paris est la capitale et la plus grande ville de France.",
        "Le bœuf bourguignon est un plat traditionnel français."
    ),
    (
        "Comment fonctionne un moteur électrique ?",
        "Un moteur électrique convertit l'énergie électrique en énergie mécanique via un champ magnétique.",
        "Les voitures thermiques utilisent de l'essence ou du diesel."
    ),
    (
        "Quels sont les symptômes du diabète ?",
        "Les symptômes incluent une soif excessive, une fatigue et des urinations fréquentes.",
        "Le Tour de France est une course cycliste annuelle."
    )
]

print(f"{'Query':<45} {'sim(+)':>7} {'sim(-)':>7} {'margin':>8}")
print("-" * 75)

for query, pos, neg in pairs:
    eq = model.encode([query])
    ep = model.encode([pos])
    en = model.encode([neg])

    sim_pos = (eq @ ep.T).item()
    sim_neg = (eq @ en.T).item()
    margin  = sim_pos - sim_neg

    print(f"{query[:43]:<45} {sim_pos:>7.4f} {sim_neg:>7.4f} {margin:>8.4f}")

## 9. Sauvegarde finale du modèle Transformers

On sauvegarde `model.model` (le `AutoModel`) et le tokenizer  :  format standard HuggingFace,  
compatible avec `AutoModel.from_pretrained()` et l'évaluation MTEB.

In [ ]:
SAVE_PATH = "./f2llm-v2-330M-lora-fr"

# Sauvegarde uniquement les poids LoRA (~10 MB au lieu de 600 MB)
model.model.save_pretrained(SAVE_PATH)
model.tokenizer.save_pretrained(SAVE_PATH)

print(f"✅ Adaptateurs LoRA sauvegardés dans : {SAVE_PATH}")
for f in sorted(os.listdir(SAVE_PATH)):
    size = os.path.getsize(os.path.join(SAVE_PATH, f))
    print(f"  {f:<40} {size/1e6:.2f} MB")

## 10. Rechargement et vérification

In [ ]:
# Recharger depuis le dossier sauvegardé
model_reloaded = F2LLMEmbedder(SAVE_PATH).to(DEVICE)
model_reloaded.eval()

test_texts = ["Bonjour le monde", "Intelligence artificielle"]

emb_original = model.encode(test_texts)
emb_reloaded = model_reloaded.encode(test_texts)

cos = F.cosine_similarity(emb_original.float(), emb_reloaded.float(), dim=1)
print(f"Cosine sim original vs reloaded : {cos}")
# Doit être tensor([1.0, 1.0]) → rechargement parfait

## 11. Évaluation MTEB French

On évalue sur un sous-ensemble représentatif des tâches MTEB French  
avant de lancer l'évaluation complète (qui prend plusieurs heures).

In [ ]:
# pip install mteb si pas déjà installé
import mteb

# Wrapper MTEB-compatible autour de F2LLMEmbedder
# MTEB attend : model.encode(sentences) -> np.ndarray
class MTEBWrapper:

    def __init__(self, embedder: F2LLMEmbedder):
        self.embedder = embedder
        self.embedder.eval()

    def encode(self, sentences, batch_size=32, **kwargs):
        all_embs = []
        for i in range(0, len(sentences), batch_size):
            chunk = sentences[i:i+batch_size]
            emb   = self.embedder.encode(chunk)
            all_embs.append(emb.float().cpu())
        return torch.cat(all_embs, dim=0).numpy()


mteb_model = MTEBWrapper(model_reloaded)

# Tâches French représentatives (subset rapide)
TASKS_FR = [
    "MintakaRetrieval",
    "MLSUMClusteringS2S",
    "MasakhaNEWSClassification",
]

tasks = mteb.get_tasks(tasks=TASKS_FR, languages=["fra"])

evaluation = mteb.MTEB(tasks=tasks)
results    = evaluation.run(
    mteb_model,
    output_folder="./mteb_results_finetuned",
    verbosity=1
)

print("\n--- Résultats MTEB French (fine-tuned) ---")
for r in results:
    print(f"{r.task_name:<35} : {r.get_score():.4f}")

In [ ]:
# Comparaison baseline vs fine-tuné
model_base = F2LLMEmbedder("codefuse-ai/F2LLM-v2-330M").to(DEVICE)
mteb_base  = MTEBWrapper(model_base)

evaluation_base = mteb.MTEB(tasks=tasks)
results_base    = evaluation_base.run(
    mteb_base,
    output_folder="./mteb_results_base",
    verbosity=1
)

# Tableau comparatif
base_scores = {r.task_name: r.get_score() for r in results_base}
ft_scores   = {r.task_name: r.get_score() for r in results}

rows = []
for task_name in ft_scores:
    base = base_scores.get(task_name, float("nan"))
    ft   = ft_scores[task_name]
    rows.append({
        "task"     : task_name,
        "baseline" : round(base, 4),
        "finetuned": round(ft,   4),
        "delta"    : round(ft - base, 4)
    })

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))